# Ensemble Learning for Breast Cancer Detection
### A High-Accuracy Model Integrating Clinical Insights — implementation

Official code for the paper published at the 2024 5th International Conference on Innovative Computing (ICIC), IEEE. DOI: [10.1109/ICIC63915.2024.11116335](https://doi.org/10.1109/ICIC63915.2024.11116335)

This notebook implements the method described in the paper:

1. **Data** — Breast Cancer Wisconsin (Diagnostic) dataset, 569 samples, 30 FNA features (357 benign, 212 malignant).
2. **Preprocessing** — EDA, mean imputation, z-score standardisation, 70 / 15 / 15 train / validation / test split.
3. **Feature selection** — correlation analysis + preliminary Random Forest feature importance.
4. **Models** — Logistic Regression, Decision Tree, Random Forest tuned with Grid Search (5-fold CV, AUC-ROC scoring) using the grids in Table I.
5. **Ensemble** — soft voting (average of predicted probabilities).
6. **Evaluation** — accuracy, sensitivity, specificity, AUC-ROC (Table II), ROC curve, feature importance (Table III), SHAP, comparison (Table IV), nested CV (Table V), threshold sensitivity analysis (Table VI), learning curves (Figs. 3–4).

> Label convention: **1 = malignant (positive class), 0 = benign**.

## 0. Setup

Install the dependencies once from the repository root with `pip install -r requirements.txt` (includes SHAP). Figures are written to `results/`, relative to this notebook's working directory.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     cross_validate, learning_curve)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, roc_curve, confusion_matrix,
                             f1_score, balanced_accuracy_score, classification_report,
                             ConfusionMatrixDisplay)

SEED = 42
import os; os.makedirs("results", exist_ok=True)
np.random.seed(SEED)
sns.set_theme(style="darkgrid")
pd.set_option("display.precision", 4)

## 1. Data collection

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data.copy()
# sklearn encodes 0 = malignant, 1 = benign -> flip so that malignant is the positive class
y = pd.Series(1 - data.target.values, name="diagnosis")

print("Shape:", X.shape)
print("Benign (0):", (y == 0).sum(), "| Malignant (1):", (y == 1).sum())
X.head()

## 2. Exploratory data analysis

In [ ]:
X.describe().T

In [ ]:
print("Missing values per column (total):", X.isna().sum().sum())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
y.map({0: "Benign", 1: "Malignant"}).value_counts().plot.bar(ax=ax[0], color=["#4C72B0", "#C44E52"])
ax[0].set_title("Class distribution"); ax[0].set_ylabel("Count")

target_corr = X.corrwith(y).sort_values()
target_corr.plot.barh(ax=ax[1], figsize=(12, 7))
ax[1].set_title("Correlation of each feature with malignancy")
plt.tight_layout(); plt.show()

In [ ]:
mean_cols = [c for c in X.columns if c.startswith("mean")]
plt.figure(figsize=(9, 7))
sns.heatmap(X[mean_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True, cbar_kws={"shrink": .8})
plt.title("Correlation between the 10 'mean' features")
plt.tight_layout(); plt.show()

## 3. Train / validation / test split (70 / 15 / 15)
Stratified so each split keeps the benign/malignant ratio. The validation set is used for model selection and threshold analysis; the test set is touched only for the final evaluation.

In [ ]:
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

for name, ys in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"{name:<11} n={len(ys):>3}  benign={(ys==0).sum():>3}  malignant={(ys==1).sum():>3}")

## 4. Feature selection layer
Combines **correlation analysis** (absolute correlation with the target) and **feature importance from a preliminary Random Forest**. Each feature gets the average of its two ranks, and the top `N_FEATURES` are kept. Implemented as a scikit-learn transformer so it is fitted on training folds only (no data leakage).

In [ ]:
N_FEATURES = 15  # number of attributes kept out of 30

class CorrRFSelector(BaseEstimator, TransformerMixin):
    # Select features by the average rank of |corr(feature, y)| and preliminary RF importance.
    def __init__(self, k=15, random_state=SEED):
        self.k = k
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X); y = np.asarray(y)
        corr = np.abs([np.corrcoef(X[:, j], y)[0, 1] for j in range(X.shape[1])])
        rf = RandomForestClassifier(n_estimators=100, random_state=self.random_state).fit(X, y)
        self.importances_ = rf.feature_importances_
        self.corr_ = corr
        corr_rank = pd.Series(corr).rank(ascending=False)
        imp_rank = pd.Series(self.importances_).rank(ascending=False)
        self.score_ = ((corr_rank + imp_rank) / 2).values
        self.support_ = np.zeros(X.shape[1], dtype=bool)
        self.support_[np.argsort(self.score_)[: self.k]] = True
        return self

    def transform(self, X):
        return np.asarray(X)[:, self.support_]

# Fit once on the training data to inspect which features are chosen
_prep = Pipeline([("imp", SimpleImputer(strategy="mean")), ("sc", StandardScaler())]).fit(X_train)
selector = CorrRFSelector(k=N_FEATURES).fit(_prep.transform(X_train), y_train)
selected_features = X.columns[selector.support_].tolist()

sel_table = pd.DataFrame({"|corr|": selector.corr_, "RF importance": selector.importances_,
                          "avg rank": selector.score_, "selected": selector.support_},
                         index=X.columns).sort_values("avg rank")
print(f"{len(selected_features)} selected features:\n", selected_features)
sel_table

## 5. Model architecture
`Input (30 features) -> mean imputation -> z-score standardisation -> feature selection -> model`

Hyper-parameter grids follow **Table I** of the paper. Grid Search uses 5-fold stratified CV with **AUC-ROC** as the scoring metric.

In [ ]:
def make_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("selector", CorrRFSelector(k=N_FEATURES)),
        ("model", model),
    ])

base_models = {
    "Logistic Regression": (
        LogisticRegression(solver="lbfgs", penalty="l2", max_iter=1000, random_state=SEED),
        {"model__C": [0.001, 0.01, 0.1, 1, 10, 100]},
    ),
    "Decision Tree": (
        DecisionTreeClassifier(criterion="gini", min_samples_split=2, min_samples_leaf=1, random_state=SEED),
        {"model__max_depth": [5, 10, 15, 20, None]},
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=100, criterion="gini", min_samples_split=2,
                               min_samples_leaf=1, bootstrap=True, random_state=SEED),
        {"model__max_depth": [10, 20, 30, None]},
    ),
}

inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

## 6. Hyper-parameter tuning (Grid Search, 5-fold CV, AUC-ROC)

In [ ]:
tuned = {}
for name, (est, grid) in base_models.items():
    gs = GridSearchCV(make_pipeline(est), grid, scoring="roc_auc", cv=inner_cv, n_jobs=-1)
    gs.fit(X_train, y_train)
    tuned[name] = gs.best_estimator_
    print(f"{name:<20} best params: {gs.best_params_}   mean CV AUC = {gs.best_score_:.4f}")

## 7. Soft-voting ensemble
Final prediction = average of the predicted malignancy probabilities of the three tuned models. Preprocessing and feature selection are shared, so the ensemble re-uses the tuned hyper-parameters.

In [ ]:
best = {n: p.named_steps["model"].get_params() for n, p in tuned.items()}

def make_ensemble(lr_params=None, dt_params=None, rf_params=None):
    lr = LogisticRegression(**(lr_params or best["Logistic Regression"]))
    dt = DecisionTreeClassifier(**(dt_params or best["Decision Tree"]))
    rf = RandomForestClassifier(**(rf_params or best["Random Forest"]))
    voting = VotingClassifier([("lr", lr), ("dt", dt), ("rf", rf)], voting="soft")
    return make_pipeline(voting)

ensemble = make_ensemble().fit(X_train, y_train)
models = {**tuned, "Ensemble (Proposed)": ensemble}

## 8. Evaluation — validation and test sets (Table II)

In [ ]:
def evaluate(model, X_, y_, threshold=0.5):
    proba = model.predict_proba(X_)[:, 1]
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_, pred).ravel()
    return {
        "Accuracy": accuracy_score(y_, pred),
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "F1": f1_score(y_, pred),
        "AUC-ROC": roc_auc_score(y_, proba),
    }

val_results = pd.DataFrame({n: evaluate(m, X_val, y_val) for n, m in models.items()}).T
test_results = pd.DataFrame({n: evaluate(m, X_test, y_test) for n, m in models.items()}).T

fmt = {"Accuracy": "{:.2%}", "Sensitivity": "{:.2%}", "Specificity": "{:.2%}", "F1": "{:.3f}", "AUC-ROC": "{:.3f}"}
print("Validation set"); display(val_results.style.format(fmt))
print("Test set (Table II)"); display(test_results.style.format(fmt))

In [ ]:
ens_test_pred = ensemble.predict(X_test)
print(classification_report(y_test, ens_test_pred, target_names=["Benign", "Malignant"], digits=4))
ConfusionMatrixDisplay.from_predictions(y_test, ens_test_pred, display_labels=["Benign", "Malignant"], cmap="Blues")
plt.title("Ensemble — confusion matrix (test set)"); plt.grid(False); plt.show()

## 9. ROC curve

In [ ]:
plt.figure(figsize=(7, 6))
for name, m in models.items():
    proba = m.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    lw = 3 if "Ensemble" in name else 1.5
    plt.plot(fpr, tpr, lw=lw, label=f"{name} (AUC = {roc_auc_score(y_test, proba):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC Curves — test set")
plt.legend(loc="lower right"); plt.tight_layout()
plt.savefig("results/fig_roc.png", dpi=300); plt.show()

## 10. Feature importance from the Random Forest (Table III)

In [ ]:
rf_pipe = tuned["Random Forest"]
rf_feats = X.columns[rf_pipe.named_steps["selector"].support_]
importance = pd.Series(rf_pipe.named_steps["model"].feature_importances_, index=rf_feats).sort_values(ascending=False)

print("Top 5 most important features:")
display(importance.head(5).rename("Importance").to_frame())

importance.sort_values().plot.barh(figsize=(8, 6), color="#4C72B0")
plt.title("Random Forest feature importance (selected features)")
plt.xlabel("Importance"); plt.tight_layout(); plt.show()

## 11. SHAP analysis (Fig. 2)

In [ ]:
try:
    import shap
    prep = Pipeline(ensemble.steps[:-1])           # imputer + scaler + selector
    voter = ensemble.named_steps["model"]
    Xtr_s = prep.transform(X_train); Xte_s = prep.transform(X_test)
    feat_names = X.columns[ensemble.named_steps["selector"].support_].tolist()

    f = lambda Z: voter.predict_proba(Z)[:, 1]     # SHAP on the full soft-voting ensemble
    explainer = shap.Explainer(f, shap.maskers.Independent(Xtr_s, max_samples=100), feature_names=feat_names)
    shap_values = explainer(Xte_s)

    shap.summary_plot(shap_values, Xte_s, feature_names=feat_names, show=False)
    plt.title("SHAP Visualization of Feature Contributions"); plt.tight_layout()
    plt.savefig("results/fig_shap.png", dpi=300, bbox_inches="tight"); plt.show()
except ImportError:
    print("SHAP is not installed. Run `pip install -r requirements.txt` and re-run this cell.")

## 12. Nested cross-validation (Table V)
Outer loop (5 folds) estimates performance; inner loop (5 folds) re-tunes the hyper-parameters of all three base models inside every outer training fold. Run on the full dataset.

In [ ]:
ens_grid = {
    "model__lr__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__dt__max_depth": [5, 10, 15, 20, None],
    "model__rf__max_depth": [10, 20, 30, None],
}
nested_base = make_ensemble(
    lr_params=dict(solver="lbfgs", penalty="l2", max_iter=1000, random_state=SEED),
    dt_params=dict(criterion="gini", min_samples_split=2, min_samples_leaf=1, random_state=SEED),
    rf_params=dict(n_estimators=100, criterion="gini", min_samples_split=2, min_samples_leaf=1,
                   bootstrap=True, random_state=SEED),
)
inner = GridSearchCV(nested_base, ens_grid, scoring="roc_auc", cv=inner_cv, n_jobs=-1)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1)

nested = cross_validate(inner, X, y, cv=outer_cv, scoring=["accuracy", "roc_auc"])  # takes a few minutes
nested_df = pd.DataFrame({"Accuracy": nested["test_accuracy"], "AUC-ROC": nested["test_roc_auc"]},
                         index=[f"Fold {i+1}" for i in range(5)])
display(nested_df)
print(f"Mean Accuracy : {nested_df['Accuracy'].mean():.2%} ± {nested_df['Accuracy'].std():.2%}")
print(f"Mean AUC-ROC  : {nested_df['AUC-ROC'].mean():.3f} ± {nested_df['AUC-ROC'].std():.3f}")

## 13. Threshold sensitivity analysis (Table VI)
The decision threshold is varied on the **validation** set; the chosen thresholds are then applied to the test set.

In [ ]:
thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
val_proba = ensemble.predict_proba(X_val)[:, 1]
test_proba = ensemble.predict_proba(X_test)[:, 1]

sweep = pd.DataFrame({
    "F1": [f1_score(y_val, val_proba >= t) for t in thresholds],
    "Balanced Accuracy": [balanced_accuracy_score(y_val, val_proba >= t) for t in thresholds],
}, index=thresholds)

t_f1, t_bal = sweep["F1"].idxmax(), sweep["Balanced Accuracy"].idxmax()
print(f"Max F1 (validation)                : {sweep['F1'].max():.3f} at threshold = {t_f1}")
print(f"Max balanced accuracy (validation) : {sweep['Balanced Accuracy'].max():.2%} at threshold = {t_bal}")
print(f"Test F1 at {t_f1}                  : {f1_score(y_test, test_proba >= t_f1):.3f}")
print(f"Test balanced accuracy at {t_bal}  : {balanced_accuracy_score(y_test, test_proba >= t_bal):.2%}")

ax = sweep.plot(figsize=(8, 4.5))
ax.axvline(t_f1, ls="--", c="C0", lw=1); ax.axvline(t_bal, ls="--", c="C1", lw=1)
ax.set_xlabel("Classification threshold"); ax.set_ylabel("Score")
ax.set_title("Threshold sensitivity analysis (validation set)")
plt.tight_layout(); plt.show()

## 14. Training and validation curves (Figs. 3 and 4)
Logistic Regression, Decision Tree and Random Forest are not trained in epochs, so the curves are plotted against the **number of training samples** (scikit-learn learning curve, 5-fold CV). Loss = log-loss of the soft-voting ensemble.

In [ ]:
sizes, tr_loss, va_loss = learning_curve(ensemble, X_train, y_train, cv=inner_cv, scoring="neg_log_loss",
                                         train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1, shuffle=True,
                                         random_state=SEED)
_, tr_acc, va_acc = learning_curve(ensemble, X_train, y_train, cv=inner_cv, scoring="accuracy",
                                   train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1, shuffle=True,
                                   random_state=SEED)

def plot_curve(train, val, ylabel, title, fname):
    plt.figure(figsize=(8, 5))
    for s, lab in [(train, "Training"), (val, "Validation")]:
        m, sd = s.mean(1), s.std(1)
        plt.plot(sizes, m, marker="o", label=f"{lab} {ylabel}")
        plt.fill_between(sizes, m - sd, m + sd, alpha=0.15)
    plt.xlabel("Number of training samples"); plt.ylabel(ylabel); plt.title(title)
    plt.legend(); plt.tight_layout(); plt.savefig(fname, dpi=300); plt.show()

plot_curve(-tr_loss, -va_loss, "Loss", "Training and Validation Loss", "results/fig_loss.png")
plot_curve(tr_acc, va_acc, "Accuracy", "Training and Validation Accuracy", "results/fig_accuracy.png")

## 15. Comparison with state-of-the-art (Table IV)

In [ ]:
ens = test_results.loc["Ensemble (Proposed)"]
comparison = pd.DataFrame([
    ["LR + DT + RF (ours, this run)", f"{ens['Accuracy']:.2%}", f"{ens['AUC-ROC']:.3f}"],
    ["Deep CNN (baseline, Akselrod-Ballin et al. [22])", "-", "0.910"],
    ["Random Forest (Anisha et al. [24])", "98%", "0.98"],
    ["Graph Neural Network (Gao et al. [25])", "97.1%", "0.989"],
], columns=["Method", "Accuracy", "AUC-ROC"])
comparison

## 16. Summary

In [ ]:
print("Soft-voting ensemble on the held-out test set")
for k, v in test_results.loc["Ensemble (Proposed)"].items():
    print(f"  {k:<12}: {v:.4f}")
print(f"\nNested CV  : accuracy {nested_df['Accuracy'].mean():.2%}, AUC-ROC {nested_df['AUC-ROC'].mean():.3f}")
print("Top-5 RF features:", ", ".join(importance.head(5).index))